Aggregations collapse many rows into summary values — `groupBy` partitions the data by one or more keys, and `agg` applies one or more aggregate functions to each group. Window functions are the next step: they compute a value **per row** using a sliding frame of neighbouring rows, without collapsing the result into one row per group.

## Setup

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.types import *

spark = (
    SparkSession.builder
    .appName("AggregationsWindowFunctions")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

DATA = os.path.join(os.path.dirname(os.path.abspath("09-aggregations-window-functions.ipynb")), "data")

card_txns = spark.read.schema(StructType([
    StructField("txn_id",      StringType(),       False),
    StructField("card_id",     StringType(),       False),
    StructField("customer_id", StringType(),       False),
    StructField("amount",      DecimalType(18, 2), False),
    StructField("merchant",    StringType(),       True),
    StructField("category",    StringType(),       True),
    StructField("txn_ts",      TimestampType(),    False),
    StructField("status",      StringType(),       False),
    StructField("is_fraud",    BooleanType(),      False),
])).parquet(f"{DATA}/card_transactions")

account_txns = spark.read.schema(StructType([
    StructField("txn_id",        StringType(),       False),
    StructField("account_id",    StringType(),       False),
    StructField("customer_id",   StringType(),       False),
    StructField("txn_type",      StringType(),       False),
    StructField("amount",        DecimalType(18, 2), False),
    StructField("balance_after", DecimalType(18, 2), False),
    StructField("description",   StringType(),       True),
    StructField("txn_ts",        TimestampType(),    False),
])).orc(f"{DATA}/account_transactions")

payments = spark.read.format("delta").load(f"{DATA}/payments")

loan_repayments = spark.read.option("header", "true").schema(StructType([
    StructField("repayment_id", StringType(),       False),
    StructField("loan_id",      StringType(),       False),
    StructField("customer_id",  StringType(),       False),
    StructField("due_date",     DateType(),         False),
    StructField("paid_date",    DateType(),         True),
    StructField("emi_amount",   DecimalType(18, 2), False),
    StructField("paid_amount",  DecimalType(18, 2), True),
    StructField("status",       StringType(),       False),
])).csv(f"{DATA}/loan_repayments")

print("Tables loaded.")

## `groupBy` + `agg`

`groupBy` partitions rows by one or more key columns. `agg` then applies aggregate functions to each partition. Both steps are lazy — execution happens when an action is called.

In [ ]:
from pyspark.sql.functions import (
    count, sum as _sum, avg, min as _min, max as _max,
    countDistinct, col, round as _round
)

# Transaction volume and spend by category
card_txns.groupBy("category").agg(
    count("txn_id").alias("num_txns"),
    _round(_sum("amount"), 2).alias("total_spend"),
    _round(avg("amount"), 2).alias("avg_txn"),
    _max("amount").alias("largest_txn"),
).orderBy(col("total_spend").desc()).show()

In [ ]:
# Multiple keys — spend per customer per category
card_txns.filter(col("status") == "SUCCESS").groupBy("customer_id", "category").agg(
    count("txn_id").alias("txns"),
    _round(_sum("amount"), 2).alias("spend"),
).orderBy("customer_id", col("spend").desc()).show(12)

In [ ]:
# Fraud rate by merchant — count fraud vs total per merchant
from pyspark.sql.functions import count, sum as _sum, round as _round, col

merchant_fraud = card_txns.groupBy("merchant").agg(
    count("txn_id").alias("total_txns"),
    _sum(col("is_fraud").cast("int")).alias("fraud_txns"),
    _round(
        _sum(col("is_fraud").cast("int")) / count("txn_id") * 100, 1
    ).alias("fraud_rate_pct"),
).orderBy(col("fraud_rate_pct").desc())

merchant_fraud.show()

In [ ]:
# Payment volume by mode — how do customers prefer to pay?
payments.filter(col("status") == "SUCCESS").groupBy("payment_mode").agg(
    count("payment_id").alias("num_payments"),
    _round(_sum("amount"), 2).alias("total_amount"),
    countDistinct("customer_id").alias("unique_customers"),
).orderBy(col("total_amount").desc()).show()

## Aggregate Function Reference

| Function | Import alias | Notes |
|---|---|---|
| `count("col")` | — | Counts non-null values; `count("*")` counts all rows |
| `sum("col")` | `_sum` | Avoids conflict with Python built-in |
| `avg("col")` | — | Mean; nulls are ignored |
| `min("col")` | `_min` | Avoids conflict with Python built-in |
| `max("col")` | `_max` | Avoids conflict with Python built-in |
| `countDistinct("col")` | — | Distinct value count |
| `approx_count_distinct` | — | Probabilistic; faster on very large data |
| `collect_list` | — | All values as an array (preserves duplicates) |
| `collect_set` | — | Unique values as an array |
| `stddev` / `variance` | — | Standard deviation / variance |
| `first` / `last` | — | First or last value in the group |

## Filtering Groups — `having` equivalent

Spark has no `having` keyword. Filter on the aggregated result by chaining `.filter()` after `.agg()`.

In [ ]:
# Customers with more than 3 successful card transactions — like SQL HAVING
high_activity = (
    card_txns
    .filter(col("status") == "SUCCESS")
    .groupBy("customer_id")
    .agg(
        count("txn_id").alias("txn_count"),
        _round(_sum("amount"), 2).alias("total_spend"),
    )
    .filter(col("txn_count") > 3)           # HAVING equivalent
    .orderBy(col("total_spend").desc())
)
high_activity.show()

## Pivot Tables

`pivot` rotates a column's distinct values into separate columns. Useful for cross-tabulation reports — for example, EMI repayment counts by status per month.

In [ ]:
from pyspark.sql.functions import count, month, col

# Repayment status counts by month — pivot status into columns
(
    loan_repayments
    .withColumn("month_num", month(col("due_date")))
    .groupBy("month_num")
    .pivot("status", ["PAID", "MISSED", "PARTIAL"])
    .agg(count("repayment_id"))
    .orderBy("month_num")
).show()

## Window Functions

![Window Function](https://raw.githubusercontent.com/schemabotview/apache-spark/main/img/spark-window-function.svg)

A window function computes a value **for each row** using a defined frame of surrounding rows. Unlike `groupBy`, the original rows are preserved — no collapse.

Three parts build a window spec:
1. `partitionBy(...)` — groups rows; the frame resets at each partition boundary
2. `orderBy(...)` — determines row order within each partition
3. Frame — `rowsBetween` or `rangeBetween` — defines which rows are included in the calculation

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, rank, dense_rank, col

# Rank card transactions by amount within each customer
w_customer_amount = (
    Window
    .partitionBy("customer_id")
    .orderBy(col("amount").desc())
)

card_txns.select(
    "customer_id", "txn_id", "amount",
    row_number().over(w_customer_amount).alias("row_num"),
    rank()        .over(w_customer_amount).alias("rank"),
    dense_rank()  .over(w_customer_amount).alias("dense_rank"),
).orderBy("customer_id", "row_num").show(12)

## Ranking Functions

| Function | Behaviour on ties |
|---|---|
| `row_number()` | Unique sequential number — ties broken arbitrarily |
| `rank()` | Tied rows share the same rank; next rank skips (1,1,3) |
| `dense_rank()` | Tied rows share the same rank; no gaps (1,1,2) |

In [ ]:
# Top 1 transaction per customer — use row_number to deduplicate
top_txn_per_customer = (
    card_txns
    .filter(col("status") == "SUCCESS")
    .withColumn("rn", row_number().over(w_customer_amount))
    .filter(col("rn") == 1)
    .select("customer_id", "txn_id", "amount", "merchant", "txn_ts")
)
top_txn_per_customer.show(8, truncate=False)

## `lead` and `lag`

`lag(col, n)` looks **back** n rows; `lead(col, n)` looks **forward** n rows within the partition. Both return `null` at the partition boundary unless a default is provided.

In [ ]:
from pyspark.sql.functions import lag, lead, col, round as _round
from pyspark.sql.window import Window

w_cust_time = Window.partitionBy("customer_id").orderBy("txn_ts")

(
    account_txns
    .withColumn("prev_amount",  lag("amount",  1).over(w_cust_time))
    .withColumn("next_amount",  lead("amount", 1).over(w_cust_time))
    .withColumn(
        "amount_delta",
        _round(col("amount") - lag("amount", 1).over(w_cust_time), 2)
    )
    .select("customer_id", "txn_ts", "txn_type", "amount", "prev_amount", "next_amount", "amount_delta")
    .orderBy("customer_id", "txn_ts")
).show(10)

## Running Aggregates — `rowsBetween`

`rowsBetween(start, end)` defines the frame in physical row positions relative to the current row:
- `Window.unboundedPreceding` — from the first row of the partition
- `Window.currentRow` — the current row
- `Window.unboundedFollowing` — to the last row of the partition

In [ ]:
from pyspark.sql.functions import sum as _sum, avg, round as _round, col
from pyspark.sql.window import Window

w_running = (
    Window
    .partitionBy("account_id")
    .orderBy("txn_ts")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

# Running total spend per account — cumulative sum from first row to current row
(
    account_txns
    .withColumn("running_total", _round(_sum("amount").over(w_running), 2))
    .select("account_id", "txn_ts", "txn_type", "amount", "balance_after", "running_total")
    .orderBy("account_id", "txn_ts")
).show(12)

In [ ]:
# 3-transaction rolling average spend per customer
w_rolling3 = (
    Window
    .partitionBy("customer_id")
    .orderBy("txn_ts")
    .rowsBetween(-2, Window.currentRow)   # current row + 2 preceding rows
)

(
    card_txns
    .filter(col("status") == "SUCCESS")
    .withColumn("rolling_avg_3", _round(avg("amount").over(w_rolling3), 2))
    .select("customer_id", "txn_ts", "amount", "rolling_avg_3")
    .orderBy("customer_id", "txn_ts")
).show(12)

## Putting It Together — Customer Payment Summary

Combine `groupBy` aggregations with window ranks to build a customer-level payment summary with ranked payment modes.

In [ ]:
from pyspark.sql.functions import count, sum as _sum, round as _round, col, row_number
from pyspark.sql.window import Window

# Step 1: total payments per customer per mode
cust_mode_totals = (
    payments
    .filter(col("status") == "SUCCESS")
    .groupBy("customer_id", "payment_mode")
    .agg(
        count("payment_id").alias("num_payments"),
        _round(_sum("amount"), 2).alias("total_amount"),
    )
)

# Step 2: rank payment modes per customer by total amount
w_cust_mode = Window.partitionBy("customer_id").orderBy(col("total_amount").desc())

preferred_mode = (
    cust_mode_totals
    .withColumn("mode_rank", row_number().over(w_cust_mode))
    .filter(col("mode_rank") == 1)   # keep each customer's top payment mode
    .select("customer_id", "payment_mode", "num_payments", "total_amount")
    .orderBy(col("total_amount").desc())
)

preferred_mode.show(10)

## Summary

- `groupBy(...).agg(...)` collapses rows into one row per group. Chain `.filter()` after `.agg()` for `HAVING`-style filtering.
- Common aggregation functions: `count`, `sum`, `avg`, `min`, `max`, `countDistinct`, `collect_list`, `collect_set`.
- `pivot` rotates a column's values into separate columns — useful for cross-tab reports.
- Window functions compute a value **per row** without collapsing — rows are preserved.
- A window spec has three parts: `partitionBy` (grouping), `orderBy` (row order), and a frame (`rowsBetween` / `rangeBetween`).
- `row_number` assigns unique sequential numbers; `rank` and `dense_rank` handle ties differently.
- `lag` / `lead` look back or forward n rows within a partition — useful for delta calculations.
- `rowsBetween(Window.unboundedPreceding, Window.currentRow)` produces a running cumulative total.